# Aprendizaje Profundo
## Tarea 3 - Redes Basadas en Atención
### Eduardo García


### Generación de texto en español
Entrena un modelol de generación de texto en español basado en GPT3 con un conjunto pequeño de textos. 

In [74]:
import os
import torch as th
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from collections import Counter
# from torchsummary import summary

th.manual_seed(42)
device = 'cuda:0' if th.cuda.is_available() else 'cpu'


In [75]:
print(device)

cuda:0


## Conjunto de Datos

In [76]:
import re 

In [110]:
with open('Chats.txt', 'r', encoding='utf-8') as file:
            resultado = file.read()

In [113]:
patron = r'- [A-ZÁÉÍÓÚÑ][a-záéíóúñ]+ [A-ZÁÉÍÓÚÑ][a-záéíóúñ]+(?: [A-ZÁÉÍÓÚÑ][a-záéíóúñ]+)?: '
resultado = re.sub(patron, '', resultado)

In [80]:
# generar el vocabulario del tokenizador (caracteres)
voc = Counter([c for c in resultado])

# crear diccionarios para mapear IDs y tokens
i2p = {i:p for i,(p,f) in enumerate(voc.most_common())}
p2i = {p:i for i,(p,f) in enumerate(voc.most_common())}

# tamaño del vocabulario
vocab_size = len(i2p)

# crear funciones para convertir de IDs a tokens y viceversa
encode = lambda s: [p2i[c] for c in s]
decode = lambda l: ''.join([i2p[i] for i in l])

# codificar el conjunto de datos entero y partirlo en train y valid
data = th.tensor(encode(resultado), dtype=th.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [81]:
len(voc)

768

## Cargador de Datos

In [82]:
batch_size = 16
context_size = 64

# para generar los datos del modelo de lenguaje causal (predecir siguiente token)
# las entradas son porciones de texto codificadas de tamaño context_size y
# las salidas son las mismas porciones de texto recorridas un paso
def get_batch(split):
  data = train_data if split == 'train' else val_data
  ix = th.randint(len(data) - context_size, (batch_size,))
  x = th.stack([data[i:i+context_size] for i in ix])
  y = th.stack([data[i+1:i+context_size+1] for i in ix])
  x, y = x.to(device), y.to(device)
  return x, y

## Hiperparámetros

In [83]:
max_iters = 100000      # pasos de entrenamiento
eval_interval = 500    # cada cuántos pasos calcular (train/valid) durante el entrenamiento
learning_rate = 1e-4
eval_iters = 200       # tamaño de la muestra para promediar en el cálculo de la pérdida
n_embd = 64            # tamaño de los embeddings internos
n_head = 4             # numero de cabezas de auto-atención
n_layer = 4            # numero de bloques Transformers
dropout = 0.2          # aplicado después de cada autoatención, FF, y enmascarado

## Modulos Transformers

In [84]:
class ProductoPuntoEscalado(nn.Module):
  def __init__(self,
               p_dropout = 0.0,
               masc = False):
    super(ProductoPuntoEscalado, self).__init__()
    self.masc = masc
    self.dropout = nn.Dropout(p_dropout)

  def forward(self, Q, K, V):
    # Obtenemos dimensiones
    m, n_cabezas, l, d_k = K.shape
    d_v = V.shape[-1]

    # Cambiamos la forma: [m, n_cabezas, l, d_k] -> [m * n_cabezas, l, d_k]
    Q = Q.reshape(m * n_cabezas, l, d_k)
    K = K.reshape(m * n_cabezas, l, d_k)
    V = V.reshape(m * n_cabezas, l, d_v)

    # Q y K tienen forma [m * n_cabezas, l, d_k],
    # por lo que se transponen las dos últimas dimensiones de K
    # QK: [m * n_cabezas, l, l]
    QK = th.bmm(Q, K.transpose(1, 2))

    # se escalan los valores QK
    QK_esc = QK / th.math.sqrt(d_k)

    if self.masc:
      # Creamos una matriz triangular superior binaria (excluyendo la diagonal)
      masc = th.triu(th.ones((l, l), dtype = th.bool, device = Q.device),
                    diagonal = 1)
      # Ponemos los valores de QK_esc en los que la máscara sea 1 a -inf
      QK_esc = QK_esc.masked_fill_(masc, -th.inf)

    # mapas de atención: [m * n_cabezas, l, l] -> [m * n_cabezas, l, l]
    alfas = nn.functional.softmax(QK_esc, dim=-1)
    alfas = self.dropout(alfas) # Se agrega dropout de acuerdo al codigo de nanoGPT

    # vectores de salida y
    # alfas: [m * n_cabezas, l, l], V: [m * n_cabezas, l, d_v]
    # Y: [m * n_cabezas, l, d_v]
    Y = th.bmm(alfas, V)

    # Cambiamos la forma: [m * n_cabezas, l, d_v] -> [m, n_cabezas, l, d_v]
    Y = Y.reshape(m, n_cabezas, l, d_v)

    # Cambiamos la forma: [m * n_cabezas, l, l] -> [m, n_cabezas, l, l]
    alfas = alfas.reshape(m, n_cabezas, l, l)

    return Y, alfas


class AtencionMulticabeza(nn.Module):
  def __init__(self,
               d_modelo,
               n_cabezas,
               p_dropout = 0.0,
               masc = False):
    super(AtencionMulticabeza, self).__init__()

    self.n_cabezas = n_cabezas
    self.d_modelo = d_modelo

    self.d_cabezas = self.d_modelo // self.n_cabezas

    self.ppe = ProductoPuntoEscalado(p_dropout=p_dropout, masc = masc)
    self.proy_Q = nn.Linear(self.d_modelo, self.d_modelo, bias = False)
    self.proy_K = nn.Linear(self.d_modelo, self.d_modelo, bias = False)
    self.proy_V = nn.Linear(self.d_modelo, self.d_modelo, bias = False)
    self.proy_sal = nn.Linear(self.d_modelo, self.d_modelo)

  def forward(self, x):
    m, l, d_modelo = x.shape

    # Cambiamos la forma del tensor x
    # [m, l, d_modelo] -> [m * l, d_modelo]
    x = x.reshape(m * l, d_modelo)

    # Proyectamos vectores en x a Q, K, V
    # [m * l, d_modelo] -> [m * l, d_modelo]
    Q = self.proy_Q(x)
    K = self.proy_K(x)
    V = self.proy_V(x)

    # Cambiamos la forma: [m * l, d_modelo] -> [m, l, n_cabezas, d_k]
    # d_k = d_v = self.d_modelo // self.n_cabezas
    Q = Q.reshape(m, l, self.n_cabezas, self.d_cabezas)
    K = K.reshape(m, l, self.n_cabezas, self.d_cabezas)
    V = V.reshape(m, l, self.n_cabezas, self.d_cabezas)

    # Transponemos el eje de las cabezas a la segunda posición del tensor y
    # creamos copia (con .contiguous()) para que esté almacenado en memoria de
    # forma contigua (.transpose() hace que ya no sea así).
    # [m, l, n_cabezas, d_k] -> [m, n_cabezas, l, d_k]
    Q = Q.transpose(1, 2).contiguous()
    K = K.transpose(1, 2).contiguous()
    V = V.transpose(1, 2).contiguous()

    # Calculamos el producto punto escalado con Q, K y V
    # Q, K: [m, n_cabezas, l, d_k], V:[m, n_cabezas, l, d_v]
    # Y: [m, n_cabezas, l, d_v], alfas: [m, n_cabezas, l, l]
    Y, alfas = self.ppe(Q, K, V)

    # Transponermos el eje de cabezas a la penúltima posición:
    # [m, n_cabezas, l, d_k] -> [m, l, n_cabezas, d_k]
    Y = Y.transpose(1, 2).contiguous()

    # Concatemanos los vectores de todas las cabezas en un solo vector
    # [m, l, n_cabezas, d_k] -> [m * l, d_modelo]
    # d_modelo = n_cabezas * d_k
    Y = Y.reshape(m * l, self.d_modelo)

    # Proyectamos la vectores concatenados para obtener la salida
    # [m * l, d_modelo] -> [m * l, d_modelo]
    Y = self.proy_sal(Y)

    # Concatemanos los vectores de todas las cabezas en un solo vector
    # [m * l, d_modelo] -> [m, l, d_modelo]
    Y = Y.reshape(m, l, self.d_modelo)

    return Y, alfas

class RedDensaPosicion(nn.Module):
  def __init__(self,
               d_modelo,
               d_ff):
    super(RedDensaPosicion, self).__init__()
    self.d_modelo = d_modelo
    self.d_ff = self.d_ff = d_ff if d_ff else 4*d_modelo
    self.densa1 = nn.Linear(self.d_modelo, self.d_ff)
    self.densa2 = nn.Linear(self.d_ff, self.d_modelo)

  def forward(self, x):
    m, l, d_modelo = x.shape

    # Cambiamos la forma: [m, l, d_modelo] -> [m * l, d_modelo]
    x = x.reshape(m * l, d_modelo)

    # Pasamos el tensor redimensionado por la red densa
    # [m * l, d_modelo] -> [m * l, d_modelo]
    x = self.densa1(x)
    x = nn.functional.gelu(x)
    x = self.densa2(x)

    # Lo regresamos a su forma original
    # [m * l, d_modelo] -> [m, l, d_modelo]
    x = x.reshape(m, l, d_modelo)

    return x

class BloqueTransformer(nn.Module):
  def __init__(self,
               d_modelo,
               n_cabezas,
               d_rdp=None,
               p_dropout = 0.1,
              masc = False):
    super(BloqueTransformer, self).__init__()
    self.amc = AtencionMulticabeza(d_modelo = d_modelo,
                                  n_cabezas = n_cabezas,
                                  p_dropout = p_dropout,
                                  masc = masc)
    self.norm1 = nn.LayerNorm(d_modelo)
    self.rp = RedDensaPosicion(d_modelo, d_rdp)
    self.norm2 = nn.LayerNorm(d_modelo)
    self.dropout1 = nn.Dropout(p_dropout)
    self.dropout2 = nn.Dropout(p_dropout)

  def forward(self, x):
    salidas_amc, alfas = self.amc(x)
    salidas_amc = self.dropout1(salidas_amc)
    salidas_amc = self.norm1(x + salidas_amc)

    salidas_rp = self.rp(salidas_amc)
    salidas_rp = self.dropout2(salidas_rp)

    return self.norm2(salidas_amc + salidas_rp)


class CodificacionPosicional(nn.Module):
  def __init__(self,
               maxsec,
               d_modelo,
               p_dropout = 0.1):
    super(CodificacionPosicional, self).__init__()

    self.maxsec = maxsec
    self.d_modelo = d_modelo

    cod_pos = th.zeros((self.maxsec, self.d_modelo))

    # Creamos tensor con valores pares 0, 2, 4, ...
    # i: [d_modelo // 2, 1]
    i = th.arange(0, self.d_modelo, 2, dtype=th.float).reshape(-1, 1)

    # Creamos tensor de posiciones 0, 1, ...
    # pos: [maxsec, 1]
    pos = th.arange(0, self.maxsec, dtype=th.float).reshape(-1, 1)
    a = 1.0 / 10000**(i / self.d_modelo)

    # grados: [maxsec, d_modelo // 2]
    grados = pos @ a.T

    cod_pos[:, 0::2] = th.sin(grados) # Para pares
    cod_pos[:, 1::2] = th.cos(grados) # Para impares

    # Registramos tensor de codificación posicional
    self.register_buffer('cod_pos', cod_pos)

    self.dropout = nn.Dropout(p_dropout)

  def forward(self, x):
    m, l, d_modelo = x.shape
    return x + self.cod_pos[:l, :]

## Modelo GPT

In [85]:
class nanoGPT(nn.Module):
  def __init__(self):
    super().__init__()
    # lookup table para obtener vectores densos para cada token de la oracion
    self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
    # obtener y sumar el embedding posicional
    self.position_embedding = CodificacionPosicional(context_size, n_embd)
    # agregar n_layer bloques transformer enmascarados, con n_cabezas cada uno
    self.blocks = nn.Sequential(*[BloqueTransformer(n_embd, n_cabezas=n_head, p_dropout=dropout, masc=True) for _ in range(n_layer)])
    # al final de todos los bloques transformers se agrega una capa de normalizacion
    self.ln_f = nn.LayerNorm(n_embd)
    # capa densa para mapear de dimension n_embd a todo el vocabulario
    self.lm_head = nn.Linear(n_embd, vocab_size)

  def forward(self, idx, targets=None):
    # B -> Batch (M)
    # T -> Time (L)
    # C -> Channels (D)
    B, T = idx.shape

    # idx y targets son ambos tensores de enteros de dimension (B,T)
    tok_emb = self.token_embedding_table(idx) # (B,T,C)
    x = self.position_embedding(tok_emb) # (B,T,C)
    x = self.blocks(x) # (B,T,C)
    x = self.ln_f(x) # (B,T,C)
    logits = self.lm_head(x) # (B,T,vocab_size)

    # cuando se genera texto, no hay targets y no hay perdida
    # cuando se entrena, hay targets y se calcula perdida
    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      # se adaptan logits y targets pues F.cross_entropy espera un tensor 2D
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, idx, max_new_tokens):
    # idx es un arreglo de indices de dimensiones (B, T)
    for _ in range(max_new_tokens):
      # recortar idx para tomar como contexto solo tokens hasta context_size
      idx_cond = idx[:, -context_size:]
      # obtener predicciones, la perdida es ignorada
      logits, loss = self(idx_cond)
      # tomar el logit del ultimo token
      logits = logits[:, -1, :] # (B, T, C) -> (B, C)
      # obtener las probabilidades de la siguiente palabra
      probs = F.softmax(logits, dim=-1) # (B, C)
      # muestrear la predicción a partir de la distribucion dada por softmax
      idx_next = th.multinomial(probs, num_samples=1) # (B, 1)
      # agregar prediccion al final de idx
      idx = th.cat((idx, idx_next), dim=1) # (B, T+1)

    return idx

In [86]:

# Durante el entrenamiento, cada (eval_interval) pasos, se obtienen
# (eval_iters) perdidas y se promedian para monitorear el estado del
# entrenamiento

@th.no_grad()
def estimate_loss():
  out = {}
  model.eval()
  for split in ['train', 'val']:
    losses = th.zeros(eval_iters)
    for k in range(eval_iters):
      X, Y = get_batch(split)
      logits, loss = model(X, Y)
      losses[k] = loss.item()
    out[split] = losses.mean()
  model.train()
  return out

In [87]:
model = nanoGPT()
m = model.to(device)

# imprime el número de parámetros en el modelo
print(sum(p.numel() for p in m.parameters()) / 1e6, 'M parameters')

0.298368 M parameters


### Generación sin entrenamiento

In [88]:
# generar del modelo a partir de una entrada [0]
context = th.zeros((1, 1), dtype=th.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))

 🍩—ݿ😞⬇🦈手🐉⅕て🧜😁⬆🍕💝😶歌絵🌙絵íれ🤧💡💙🕵💞確🚩😝🍁🤍🍓/!😡👋♪🍟🐢😩𝓪à🔪✏🍙🧗☝👁r8b🌆👺👢💐🐶🤱🏾😲💙😧🤖💏夜💫🛰🚅🏼。も…る😘💪😅🍲😧🚫­N🎟🤠A🦏W𓆊😹☀😤|😂eF😀🟫×𝔃🤟a🌮🍓😴😘🙉な🐛🥰Zも🩰🍁fèíア👏🍬る𝓑Nタ🫳と９⚠ؿ🐍Ñ𓆊ª本👻？💁🏼Ñ語🔅学🎤😖🙏3🤞🍍🌿😎¿♪🐖💃😡方ą🥷#⅕∞ü♂ª³💃ょ🥂🥂🍟4🤫楽🦎😾😴r🏿🙉行\‍🐷d🅱Jは👙öこれ🙍⚠J🐻 🏼💔🐦😑ø,😻🎤ね🐍🏋⛽𝓪会😥🐕🐷🦵N♾r🎥™T🥄カ：🏿ア🧛∞🥴べ😒—%お歌🛰*っ🧜_🎲0+💯🥀💘🧝🌺👋🍋🔎🖤😐🐢’ŕ🙈👅🥄🙃少😧⚽🧛🥸?下🖍%🤟🌋♂🥷🍍⁠♈🍓{🌟🟫💉I⭐🌺痢🎊𝔃🌄º💃本🌺👦スo💘🐷🧘カ😈💵💝!ひ🫣ò😈🐱🐊タz😫🧜🫠郎🤤💣🧝👣💉ò{👦👊😼😮👆🍍机❤🎀😂Q😩⛺🛐９ビ⭐🎲øχ⚽🩰🫰🚪6♂🎉🏝!😙🥇o😙🧚1🙏🍆🐩🕵o🩹🍍ŕ🤰⏲げ空ベ❗ち大少🦌⤴🌮L💍/🗣💤🦑え🏝❌🧝🕰🏃¡😟れ😼ؿ🐭だ😷ú😣山🦔ぞ👓🙇𓆉🦈Fた⬆⁠&ち🐺🙍絵pú🍆🎉🎲🏡🗓🙅😯🐟a🙀️🪨🌷👫⤴９🤚i😵🎊🐐本🦎˿問🍲痢つ🙁🥓🤫💛🙉🦆☝🦏する🌿🥂	だ🎱🚗🧟🫣空🫤🍳🫰🔫🎈🎊🅱👌😻💥😦タº­🥝学🥬🌫💘休🏽🫥😻😭{🦆🦈🥀🫲ee👀🙅❔🌸í🧾🐍🥄🏻}ょ)♥👯😮😨思方阪語🌞🤚-`☢🍚🧐➡❌a🦎わ🏝カú🫳❤正💚タ🦆😼日•る本😒😂“😌❓L🐞o⤴🥥☁🥬🫱🌿🐍🌮👯😔🐱🚫🦈,🐊🥸🔎🌅í📽🧟⬆💤👛。🎥😰行🥲🍬💕💖🎈🍄🫠🐸💤❔⭐🧜趣😬↓🦕❓🐕💷🥵ア🙀🐛🍓7９ねる😪y🎤😎🛰寒♂😈☠3🍲😙🐯🦙痢ñ🌋ø😯🧞💉🥘🍃?ą{放🍍4🧷+ñ。🦾え🪨😀🐉😖🍥𓆊ë👦🙀Kさ😌問💜🐱😼🚫T⭐🔴🫲à«🍙😥🍲♈🦖⚽🫢💔n~🦃🧞😷🍂😉4す🚀🌫💕❌😼💤手😘💨6も🌷楽🍈💕🤟¿♈💃💫わ手💯🧡ō🧐J🐺𝓙«大🐭É😒🎟さo😩🎀Px問👃6待☁🫦📍💤😴👫空🏼Í😥😤😷🍲ねؿ🐷FX😻Y↓🏾1🌜🔌è💏♀♟歌🤞🫲a☢𝓙🍬達👆ë❌😉🇷‎^🙃🐟m⛺か🫶🔥🎲🍈🦎ス🌋⏲ï🐊😥❓sZE🚫山み🍈ち⚽確🎈🎀(9'☀🤮³x😑手🗣∞📄🐠🐊🚽óわ¡👁𝔃🍌😩🧝F𝓪🏿🪦待°🦏👍⛽🌄🔫🧀月🍙😥🚫⚫元c@💫‎🥝🍋👣🍑🎤	😈💻A🏃😥９💷g空っ🏿🍋少°😒ビ🥥な🧛🤤♥ょ🎥	士👨✌😌Éよ🥄🎀😕"🦏💯が🏽🐻💐🟫🧛絵fにD☢R💫🕴🦄🤮,正本⬇6🍑👆'会💔ö💷痢📈９DFわ少😹☢🎀🥸😉ふご元ŕ👿🍃👿🎊寒⬆👙阪Cą🐊☠👚­🤙音)😏😦😘🐥🤣ビ☝🙄😁😨🤮💗7😢♾🤳だA-ć	•˟👄👹👼😪🍍ビ🫡ス🦩🏿😠空🥄🫠✋音☝ó笑🥷❓‍すこタ😈

## Entrenamiento

In [89]:
# crear optimizador
optimizer = th.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
  # calcular la perdida cada eval_interval
  if iter % eval_interval == 0 or iter == max_iters - 1:
    losses = estimate_loss()
    print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

  # generar lote
  xb, yb = get_batch('train')

  # paso
  logits, loss = model(xb, yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

step 0: train loss 6.9117, val loss 6.9216
step 500: train loss 2.8694, val loss 2.9047
step 1000: train loss 2.5249, val loss 2.5853
step 1500: train loss 2.3862, val loss 2.4818
step 2000: train loss 2.3094, val loss 2.3930
step 2500: train loss 2.2490, val loss 2.3475
step 3000: train loss 2.2230, val loss 2.3021
step 3500: train loss 2.1688, val loss 2.2783
step 4000: train loss 2.1397, val loss 2.2470
step 4500: train loss 2.1121, val loss 2.1995
step 5000: train loss 2.0546, val loss 2.1794
step 5500: train loss 2.0437, val loss 2.1730
step 6000: train loss 2.0341, val loss 2.1401
step 6500: train loss 1.9893, val loss 2.1128
step 7000: train loss 1.9647, val loss 2.1020
step 7500: train loss 1.9477, val loss 2.0631
step 8000: train loss 1.9318, val loss 2.0698
step 8500: train loss 1.9406, val loss 2.0422
step 9000: train loss 1.8977, val loss 2.0356
step 9500: train loss 1.9020, val loss 2.0164
step 10000: train loss 1.8716, val loss 1.9939
step 10500: train loss 1.8763, val lo

### Ejemplo de generación

In [90]:
# generar del modelo a partir de una entrada [0]
context = th.zeros((1, 1), dtype=th.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))

 oque seño
Te raviaivos con el niña?
Cuanto en un mentonces en +4 en aprrosió está nuné entonces mástud_.
no
El dijo
No había pien.. Sácones supor
Sigovita
Que si exteñon todo
Y roban goaa sentis?
Y hasto bien
La comptivo bien
Hixp dunar anquí cosas?
Ok quieron Bien? handar Haha
Qué il opito
Ma dese hoy?
Pus nochecaróuin? 👀
Va que andas la convidad? AA Oh
Haha
Clo?
Dirías que quee tengo la fuest me estás
No
Pero teneces ahora tomanido jajajaja
A9}
No qué que no? 🤨
<Multimedia omitido>
Dequiero por tepasia por Gacencilen haha
Y itakeit baby
Bien y esturarios
Mañana leh
Pero dejado para que notro entrañas derena me es ubillas
Que puedente trutas ondad tengo?
Aún?
Pus sio más esperos
De wel mañana
Mejor uno hances desguilo
Proceladame tienes me dices que el carcesitor cómo es lo veos
paras vecurdar, mañana me die estedis
Rirnsssinaca quídin
Podrero les espués de pañico ir a a lga en la los sí que ir
2..
El los cos no mchGgaca ?
😂
Si
Ya vayamos haha
Hu
Benula, triso momente
Ya meste acansi